<a href="https://colab.research.google.com/github/mahdad277/repo1/blob/process-kaggle-datasets/Simulated_Movie_Ingestion_and_SQL_Query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import sqlite3
import re
import io
from itertools import chain

# Import the Colab-specific file handling library
try:
    from google.colab import files
except ImportError:
    print("Warning: 'google.colab' not found. This script is intended for Google Colab.")
    # Define dummy functions for non-colab environments for testing purposes
    class DummyFiles:
        def upload(self):
            print("Please provide a file named 'sample_data.csv' in the current directory for local testing.")
            return {'sample_data.csv': b'id,title,votes,release_date\r\ntt0027483,"The Crimson Circle",30,1936-08-10\r\ntt0058131,"The Mystery of Thug Island",114,1966-05-28\r\n'}
    files = DummyFiles()

def run_data_pipeline():
    """
    Handles file upload, data transformation, SQL ingestion, and query execution.
    """

    print("Please upload your movie dataset file (CSV format) now.")

    # 1. FILE UPLOAD AND READING
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Exiting.")
        return

    # Get the first uploaded file name
    file_name = next(iter(uploaded))
    print(f"Successfully uploaded: {file_name}")

    try:
        # Read the uploaded file into a DataFrame
        df_raw = pd.read_csv(io.StringIO(uploaded[file_name].decode('utf-8')))
        ROW_LIMIT = 200 # Process only the top N rows
        df_raw = df_raw.head(ROW_LIMIT)
        print(f"Loaded {len(df_raw)} rows from the dataset for processing.")

    except Exception as e:
        print(f"Error reading the file: {e}")
        return

    # --- 2. DATA TRANSFORMATION: Creating the required 'movies' table ---

    # Original SQL requirements: movies table needs 'id', 'name', 'year'

    movies_df = df_raw[['id', 'title', 'release_date']].copy()

    # Rename the columns to match the SQL query's 'movies' table schema
    movies_df = movies_df.rename(columns={
        'id': 'id',        # Maps to movies.id
        'title': 'name'    # Maps to movies.name
    })

    # Extract the year from the 'release_date' column
    movies_df['year'] = pd.to_datetime(movies_df['release_date'], errors='coerce').dt.year
    # Fill missing years with 0 and convert to integer
    movies_df['year'] = movies_df['year'].fillna(0).astype(int)

    # Final clean 'movies' table structure
    movies_df = movies_df[['id', 'name', 'year']]
    print(f"\nPrepared 'movies' table with {len(movies_df)} unique movies.")


    # --- 3. DATA TRANSFORMATION: Creating the required 'ratings' table ---

    # Original SQL requirements: ratings table needs 'movie_id' (one row per rating)
    # We simulate this by repeating the movie ID based on the 'votes' count.

    # Ensure 'votes' is treated as a number, defaulting to 0 for non-numeric/missing values
    df_raw['votes'] = pd.to_numeric(df_raw['votes'], errors='coerce').fillna(0).astype(int)

    # 3.1 Get a list of movie IDs repeated by their vote count
    expanded_movie_ids = []
    for index, row in df_raw[['id', 'votes']].iterrows():
        # Append the movie ID 'id' as many times as 'votes'
        expanded_movie_ids.extend([row['id']] * row['votes'])

    # 3.2 Create the final 'ratings' DataFrame
    ratings_df = pd.DataFrame({'movie_id': expanded_movie_ids})

    print(f"Prepared 'ratings' table with {len(ratings_df)} synthetic rating entries.")


    # --- 4. INGEST DATA INTO SQLITE DATABASE ---

    # Use an in-memory SQLite database connection
    conn = sqlite3.connect(':memory:')

    # Ingest the two prepared DataFrames into SQL tables named 'movies' and 'ratings'
    movies_df.to_sql('movies', conn, if_exists='replace', index=False)
    ratings_df.to_sql('ratings', conn, if_exists='replace', index=False)


    # --- 5. EXECUTE THE ORIGINAL SQL QUERY ---

    # Your original SQL query, now running against the consistent tables
    sql_query = """
    SELECT
        r.movie_id,
        m.name,
        COUNT(*) AS times_rated
    FROM
        ratings r
    JOIN
        movies m ON r.movie_id = m.id
    GROUP BY
        r.movie_id, m.name, m.year
    ORDER BY
        times_rated DESC
    LIMIT
        200
    """

    print("\n--- Executing SQL Query ---")

    # Execute the query and load the results into a new DataFrame
    most_rated_movies_df = pd.read_sql_query(sql_query, conn)

    # --- 6. DISPLAY RESULTS AND CLEANUP ---
    print("Query Results (Most Rated Movies):")
    print(most_rated_movies_df)

    conn.close()
    print("\nProcess complete.")

# Run the pipeline function
run_data_pipeline()


Please upload your movie dataset file (CSV format) now.


Saving final_dataset.csv to final_dataset.csv
Successfully uploaded: final_dataset.csv
Loaded 200 rows from the dataset for processing.

Prepared 'movies' table with 200 unique movies.
Prepared 'ratings' table with 38638 synthetic rating entries.

--- Executing SQL Query ---
Query Results (Most Rated Movies):
     movie_id                              name  times_rated
0   tt0889588         The Children of Huang Shi        10000
1   tt9133378                         Premature          997
2   tt0107492  The Making of '...and God Spoke'          964
3   tt0042874              Radar Secret Service          954
4   tt3790720      Peggy Guggenheim: Art Addict          889
..        ...                               ...          ...
93  tt0015338             The Sixth Commandment           16
94  tt0016223                   The Love Pirate           16
95  tt0014515            Strangers of the Night           13
96  tt0183572   Nessa no chikai (Zenpen; Kôhen)           11
97  tt0169615     